# 00 - Data Preprocessing Pipeline



---

This notebook preprocesses the **MAESTRO v3.0.0** piano MIDI dataset for downstream
music-generation tasks. It performs the following steps:

1. Dataset exploration and statistics
2. MIDI parsing and note-event extraction
3. Piano-roll conversion (2-D matrix representation)
4. Sequence windowing for autoregressive / VAE models
5. Token-based event encoding for Transformer models

All outputs are saved under `outputs/` for consumption by the training notebooks.

---
## Section 1 — Setup & Imports

In [ ]:
# ── Install any missing packages (safe to re-run) ──
import subprocess, sys

for pkg in ["pretty_midi", "music21", "numpy", "pandas",
            "matplotlib", "seaborn", "torch", "tqdm"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import pretty_midi
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
print(f"NumPy  {np.__version__}  |  Pandas {pd.__version__}")

In [ ]:
# ── Paths ──
MIDI_ROOT   = Path("../maestro-v3.0.0")
CSV_PATH    = MIDI_ROOT / "maestro-v3.0.0.csv"
OUTPUT_ROOT = Path("outputs")

OUTPUT_DIRS = [
    OUTPUT_ROOT / "processed",
    OUTPUT_ROOT / "plots",
    OUTPUT_ROOT / "task1" / "generated_midis",
    OUTPUT_ROOT / "task2" / "generated_midis",
    OUTPUT_ROOT / "task3" / "generated_midis",
    OUTPUT_ROOT / "task4" / "generated_midis",
]

for d in OUTPUT_DIRS:
    os.makedirs(d, exist_ok=True)
    print(f"  ✓ {d}")

print("\nAll output directories ready.")

---
## Section 2 — Dataset Exploration

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"CSV loaded: {len(df)} rows, columns = {list(df.columns)}")
df.head()

In [ ]:
print("=" * 55)
print("           MAESTRO v3.0.0  —  Dataset Summary")
print("=" * 55)
print(f"  Total MIDI files    : {len(df)}")
print(f"  Total duration (hr) : {df['duration'].sum() / 3600:.2f}")
print(f"  Mean duration (min) : {df['duration'].mean() / 60:.2f}")
print(f"  Median duration (s) : {df['duration'].median():.1f}")
print(f"  Years present       : {sorted(df['year'].unique())}")
print("=" * 55)

print("\nSplit distribution:")
print(df['split'].value_counts().to_string())

In [ ]:
# ── Plot 1: MIDI files per year ──
fig, ax = plt.subplots(figsize=(10, 5))
year_counts = df['year'].value_counts().sort_index()
sns.barplot(x=year_counts.index.astype(str), y=year_counts.values, ax=ax,
            palette="viridis")
ax.set_title("Number of MIDI Files per Year", fontsize=14, fontweight="bold")
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
for i, v in enumerate(year_counts.values):
    ax.text(i, v + 1, str(v), ha="center", fontsize=9)
plt.tight_layout()
fig.savefig(OUTPUT_ROOT / "plots" / "dataset_year_distribution.png", dpi=150)
plt.show()
print("Saved: outputs/plots/dataset_year_distribution.png")

In [ ]:
# ── Plot 2: Duration distribution ──
fig, ax = plt.subplots(figsize=(10, 5))
durations_min = df['duration'] / 60.0
ax.hist(durations_min, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(durations_min.mean(), color="red", linestyle="--", label=f"Mean = {durations_min.mean():.1f} min")
ax.axvline(durations_min.median(), color="orange", linestyle="--", label=f"Median = {durations_min.median():.1f} min")
ax.set_title("Distribution of Piece Durations", fontsize=14, fontweight="bold")
ax.set_xlabel("Duration (minutes)", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
fig.savefig(OUTPUT_ROOT / "plots" / "duration_distribution.png", dpi=150)
plt.show()
print("Saved: outputs/plots/duration_distribution.png")

In [ ]:
# ── Split counts ──
for split_name in ["train", "validation", "test"]:
    sub = df[df["split"] == split_name]
    print(f"  {split_name:>12s}:  {len(sub):>5d} files  |  "
          f"{sub['duration'].sum() / 3600:.2f} hours")

---
## Section 3 — MIDI Parser

In [ ]:
def load_midi(filepath):
    """Load a MIDI file and return note events.

    Parameters
    ----------
    filepath : str or Path
        Path to a .mid / .midi file.

    Returns
    -------
    list[tuple]
        Each element is (pitch, velocity, start_time, end_time).
        Returns an empty list if the file cannot be parsed.
    """
    try:
        pm = pretty_midi.PrettyMIDI(str(filepath))
        notes = []
        for instrument in pm.instruments:
            for note in instrument.notes:
                notes.append((note.pitch, note.velocity,
                              note.start, note.end))
        notes.sort(key=lambda x: x[2])
        return notes
    except Exception as e:
        print(f"  [WARN] Could not load {filepath}: {e}")
        return []

In [ ]:
# ── Test on 3 sample files ──
sample_rows = df.head(3)
for _, row in sample_rows.iterrows():
    fpath = MIDI_ROOT / row["midi_filename"]
    notes = load_midi(fpath)
    print(f"\n{row['midi_filename']}  —  {len(notes)} notes")
    if notes:
        print("  First 10 note events (pitch, vel, start, end):")
        for n in notes[:10]:
            print(f"    pitch={n[0]:3d}  vel={n[1]:3d}  "
                  f"start={n[2]:.3f}  end={n[3]:.3f}")

---
## Section 4 — Piano Roll Conversion

In [ ]:
def midi_to_piano_roll(filepath, fs=16):
    """Convert a MIDI file to a normalised piano-roll matrix.

    Parameters
    ----------
    filepath : str or Path
        Path to a MIDI file.
    fs : int
        Sampling frequency (time-steps per second).

    Returns
    -------
    np.ndarray or None
        Shape (128, T) with values in [0, 1], or None on failure.
    """
    try:
        pm = pretty_midi.PrettyMIDI(str(filepath))
        roll = pm.get_piano_roll(fs=fs)          # shape (128, T)
        if roll.max() > 0:
            roll = roll / roll.max()              # normalise to [0, 1]
        return roll
    except Exception as e:
        print(f"  [WARN] Piano-roll failed for {filepath}: {e}")
        return None


def piano_roll_to_midi(piano_roll, fs=16, output_path=None, velocity=100):
    """Reconstruct a MIDI file from a piano-roll matrix.

    Parameters
    ----------
    piano_roll : np.ndarray
        Shape (128, T) with values in [0, 1].
    fs : int
        Sampling frequency used to create the piano roll.
    output_path : str or Path, optional
        If provided, save the MIDI file to this path.
    velocity : int
        Default velocity for reconstructed notes.

    Returns
    -------
    pretty_midi.PrettyMIDI
    """
    pm = pretty_midi.PrettyMIDI()
    instrument = pretty_midi.Instrument(program=0, name="Piano")
    binarized = (piano_roll > 0.5).astype(int)
    dt = 1.0 / fs

    for pitch in range(128):
        in_note = False
        start = 0.0
        for t in range(binarized.shape[1]):
            if binarized[pitch, t] == 1 and not in_note:
                in_note = True
                start = t * dt
            elif binarized[pitch, t] == 0 and in_note:
                in_note = False
                end = t * dt
                note = pretty_midi.Note(velocity=velocity, pitch=pitch,
                                        start=start, end=end)
                instrument.notes.append(note)
        if in_note:
            end = binarized.shape[1] * dt
            note = pretty_midi.Note(velocity=velocity, pitch=pitch,
                                    start=start, end=end)
            instrument.notes.append(note)

    pm.instruments.append(instrument)
    if output_path is not None:
        pm.write(str(output_path))
        print(f"  Saved MIDI -> {output_path}")
    return pm

In [ ]:
# ── Plot 3: Piano-roll heatmap of one sample ──
sample_file = MIDI_ROOT / df.iloc[0]["midi_filename"]
sample_roll = midi_to_piano_roll(sample_file, fs=16)

if sample_roll is not None:
    T_display = min(sample_roll.shape[1], 800)  # first ~50 s at fs=16
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(sample_roll[:, :T_display], aspect="auto", origin="lower",
              cmap="magma", interpolation="nearest")
    ax.set_title(f"Piano Roll — {df.iloc[0]['midi_filename']} (first {T_display} frames)",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Time step", fontsize=12)
    ax.set_ylabel("MIDI Pitch", fontsize=12)
    plt.tight_layout()
    fig.savefig(OUTPUT_ROOT / "plots" / "sample_piano_roll.png", dpi=150)
    plt.show()
    print("Saved: outputs/plots/sample_piano_roll.png")
    print(f"Piano-roll shape: {sample_roll.shape}")

---
## Section 5 — Sequence Windowing

In [ ]:
def create_sequences(piano_roll, seq_len=64, stride=32):
    """Segment a piano roll into overlapping fixed-length windows.

    Parameters
    ----------
    piano_roll : np.ndarray
        Shape (128, T).
    seq_len : int
        Number of time steps per window.
    stride : int
        Step size between consecutive windows.

    Returns
    -------
    np.ndarray
        Shape (N, seq_len, 128) where N is the number of windows.
    """
    roll_T = piano_roll.T  # (T, 128)
    T = roll_T.shape[0]
    seqs = []
    for start in range(0, T - seq_len + 1, stride):
        seqs.append(roll_T[start:start + seq_len])
    if len(seqs) == 0:
        return np.empty((0, seq_len, 128), dtype=np.float32)
    return np.array(seqs, dtype=np.float32)

In [ ]:
def process_split_piano_roll(split_name, dataframe, seq_len=64, stride=32, fs=16):
    """Convert all MIDI files in a split to windowed sequences."""
    sub = dataframe[dataframe["split"] == split_name]
    all_seqs = []
    skipped = 0
    for idx, row in tqdm(sub.iterrows(), total=len(sub),
                         desc=f"Piano-roll [{split_name}]"):
        fpath = MIDI_ROOT / row["midi_filename"]
        roll = midi_to_piano_roll(fpath, fs=fs)
        if roll is None:
            skipped += 1
            continue
        seqs = create_sequences(roll, seq_len=seq_len, stride=stride)
        if seqs.shape[0] > 0:
            all_seqs.append(seqs)
    if skipped:
        print(f"  Skipped {skipped} files in '{split_name}'.")
    result = np.concatenate(all_seqs, axis=0) if all_seqs else np.empty((0, seq_len, 128))
    print(f"  {split_name}: {result.shape[0]} sequences  |  shape {result.shape}")
    return result

In [ ]:
train_seqs = process_split_piano_roll("train", df)
np.save(OUTPUT_ROOT / "processed" / "train_sequences.npy", train_seqs)
print(f"Saved train_sequences.npy — {train_seqs.shape}")

In [ ]:
val_seqs = process_split_piano_roll("validation", df)
np.save(OUTPUT_ROOT / "processed" / "val_sequences.npy", val_seqs)
print(f"Saved val_sequences.npy — {val_seqs.shape}")

test_seqs = process_split_piano_roll("test", df)
np.save(OUTPUT_ROOT / "processed" / "test_sequences.npy", test_seqs)
print(f"Saved test_sequences.npy — {test_seqs.shape}")

In [ ]:
print("\n── Sequence array shapes ──")
print(f"  Train      : {train_seqs.shape}")
print(f"  Validation : {val_seqs.shape}")
print(f"  Test       : {test_seqs.shape}")

---
## Section 6 — Token-Based Representation (for Task 3)

In [ ]:
# ── Build vocabulary ──
#   NOTE_ON_0  .. NOTE_ON_127   → 0..127   (128 tokens)
#   NOTE_OFF_0 .. NOTE_OFF_127  → 128..255  (128 tokens)
#   TIME_SHIFT_1 .. TIME_SHIFT_10 → 256..265 (10 tokens, each = 100 ms)
#   VELOCITY_0 .. VELOCITY_31   → 266..297  (32 tokens)
#   Total: 298 tokens

def build_vocab():
    """Build the event-token vocabulary.

    Returns
    -------
    dict
        Mapping from token name (str) to integer id.
    """
    vocab = {}
    idx = 0
    for p in range(128):
        vocab[f"NOTE_ON_{p}"] = idx; idx += 1
    for p in range(128):
        vocab[f"NOTE_OFF_{p}"] = idx; idx += 1
    for t in range(1, 11):
        vocab[f"TIME_SHIFT_{t}"] = idx; idx += 1
    for v in range(32):
        vocab[f"VELOCITY_{v}"] = idx; idx += 1
    return vocab

VOCAB = build_vocab()
ID2TOKEN = {v: k for k, v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)
print(f"Vocabulary size: {VOCAB_SIZE}")

In [ ]:
def tokenize_midi(filepath):
    """Convert a MIDI file into a sequence of integer tokens.

    Parameters
    ----------
    filepath : str or Path
        Path to a MIDI file.

    Returns
    -------
    list[int] or None
        Token sequence, or None on failure.
    """
    try:
        pm = pretty_midi.PrettyMIDI(str(filepath))
    except Exception as e:
        print(f"  [WARN] Tokenise failed for {filepath}: {e}")
        return None

    events = []
    for inst in pm.instruments:
        for note in inst.notes:
            events.append(("on",  note.start, note.pitch, note.velocity))
            events.append(("off", note.end,   note.pitch, 0))
    events.sort(key=lambda x: x[1])

    tokens = []
    prev_time = 0.0
    for ev_type, time, pitch, velocity in events:
        # Emit time-shift tokens
        dt = time - prev_time
        while dt > 0:
            shift = min(dt, 1.0)
            shift_bin = max(1, min(10, int(round(shift / 0.1))))
            tokens.append(VOCAB[f"TIME_SHIFT_{shift_bin}"])
            dt -= shift_bin * 0.1
        prev_time = time

        if ev_type == "on":
            vel_bin = min(31, velocity // 4)
            tokens.append(VOCAB[f"VELOCITY_{vel_bin}"])
            tokens.append(VOCAB[f"NOTE_ON_{pitch}"])
        else:
            tokens.append(VOCAB[f"NOTE_OFF_{pitch}"])

    return tokens


def detokenize(tokens, output_path=None, default_velocity=80):
    """Reconstruct a MIDI file from a token sequence.

    Parameters
    ----------
    tokens : list[int]
        Token sequence.
    output_path : str or Path, optional
        If given, write the MIDI file to this path.
    default_velocity : int
        Fallback velocity when none is specified.

    Returns
    -------
    pretty_midi.PrettyMIDI
    """
    pm = pretty_midi.PrettyMIDI()
    instrument = pretty_midi.Instrument(program=0, name="Piano")

    current_time = 0.0
    current_vel = default_velocity
    active_notes = {}  # pitch -> start_time

    for tok in tokens:
        name = ID2TOKEN.get(tok, "")
        if name.startswith("TIME_SHIFT_"):
            shift = int(name.split("_")[-1]) * 0.1
            current_time += shift
        elif name.startswith("VELOCITY_"):
            current_vel = int(name.split("_")[-1]) * 4
        elif name.startswith("NOTE_ON_"):
            pitch = int(name.split("_")[-1])
            active_notes[pitch] = current_time
        elif name.startswith("NOTE_OFF_"):
            pitch = int(name.split("_")[-1])
            if pitch in active_notes:
                start = active_notes.pop(pitch)
                end = max(current_time, start + 0.05)
                note = pretty_midi.Note(velocity=current_vel, pitch=pitch,
                                        start=start, end=end)
                instrument.notes.append(note)

    # Close any remaining active notes
    for pitch, start in active_notes.items():
        note = pretty_midi.Note(velocity=current_vel, pitch=pitch,
                                start=start, end=current_time + 0.25)
        instrument.notes.append(note)

    pm.instruments.append(instrument)
    if output_path is not None:
        pm.write(str(output_path))
        print(f"  Saved MIDI -> {output_path}")
    return pm

In [ ]:
def tokenize_split(split_name, dataframe):
    """Tokenize all MIDI files in a data split."""
    sub = dataframe[dataframe["split"] == split_name]
    all_tokens = []
    lengths = []
    skipped = 0
    for _, row in tqdm(sub.iterrows(), total=len(sub),
                       desc=f"Tokenise [{split_name}]"):
        fpath = MIDI_ROOT / row["midi_filename"]
        toks = tokenize_midi(fpath)
        if toks is None:
            skipped += 1
            continue
        all_tokens.append(toks)
        lengths.append(len(toks))
    if skipped:
        print(f"  Skipped {skipped} files in '{split_name}'.")
    print(f"  {split_name}: {len(all_tokens)} files tokenised  |  "
          f"mean length = {np.mean(lengths):.0f} tokens")
    return all_tokens

In [ ]:
train_tokens = tokenize_split("train", df)
np.save(OUTPUT_ROOT / "processed" / "train_tokens.npy",
        np.array(train_tokens, dtype=object), allow_pickle=True)
print(f"Saved train_tokens.npy — {len(train_tokens)} sequences")

In [ ]:
val_tokens = tokenize_split("validation", df)
np.save(OUTPUT_ROOT / "processed" / "val_tokens.npy",
        np.array(val_tokens, dtype=object), allow_pickle=True)
print(f"Saved val_tokens.npy — {len(val_tokens)} sequences")

test_tokens = tokenize_split("test", df)
np.save(OUTPUT_ROOT / "processed" / "test_tokens.npy",
        np.array(test_tokens, dtype=object), allow_pickle=True)
print(f"Saved test_tokens.npy — {len(test_tokens)} sequences")

In [ ]:
# ── Save vocabulary mapping ──
vocab_path = OUTPUT_ROOT / "processed" / "vocab.json"
with open(vocab_path, "w") as f:
    json.dump(VOCAB, f, indent=2)
print(f"Saved vocabulary -> {vocab_path}  ({VOCAB_SIZE} tokens)")

# ── Show sample token sequence ──
if train_tokens:
    sample_toks = train_tokens[0][:30]
    print("\nSample token IDs (first 30):")
    print(sample_toks)
    print("\nDecoded:")
    for t in sample_toks:
        print(f"  {t:3d} -> {ID2TOKEN[t]}")

---
## Section 7 — Summary

In [ ]:
print("=" * 65)
print("          Preprocessing Pipeline — Output Summary")
print("=" * 65)

output_files = [
    ("train_sequences.npy",  OUTPUT_ROOT / "processed" / "train_sequences.npy"),
    ("val_sequences.npy",    OUTPUT_ROOT / "processed" / "val_sequences.npy"),
    ("test_sequences.npy",   OUTPUT_ROOT / "processed" / "test_sequences.npy"),
    ("train_tokens.npy",     OUTPUT_ROOT / "processed" / "train_tokens.npy"),
    ("val_tokens.npy",       OUTPUT_ROOT / "processed" / "val_tokens.npy"),
    ("test_tokens.npy",      OUTPUT_ROOT / "processed" / "test_tokens.npy"),
    ("vocab.json",           OUTPUT_ROOT / "processed" / "vocab.json"),
    ("dataset_year_distribution.png", OUTPUT_ROOT / "plots" / "dataset_year_distribution.png"),
    ("duration_distribution.png",     OUTPUT_ROOT / "plots" / "duration_distribution.png"),
    ("sample_piano_roll.png",         OUTPUT_ROOT / "plots" / "sample_piano_roll.png"),
]

print(f"{'File':<35s}  {'Exists':>6s}  {'Size':>12s}")
print("-" * 58)
for name, path in output_files:
    exists = path.exists()
    size_str = ""
    if exists:
        size_bytes = path.stat().st_size
        if size_bytes < 1024:
            size_str = f"{size_bytes} B"
        elif size_bytes < 1024 ** 2:
            size_str = f"{size_bytes / 1024:.1f} KB"
        else:
            size_str = f"{size_bytes / 1024**2:.1f} MB"
    print(f"  {name:<33s}  {'yes' if exists else 'NO':>6s}  {size_str:>12s}")

print("\n── Sequence shapes ──")
print(f"  Train sequences : {train_seqs.shape}")
print(f"  Val sequences   : {val_seqs.shape}")
print(f"  Test sequences  : {test_seqs.shape}")

print(f"\n── Token counts ──")
print(f"  Train files tokenised : {len(train_tokens)}")
print(f"  Val files tokenised   : {len(val_tokens)}")
print(f"  Test files tokenised  : {len(test_tokens)}")
print(f"  Vocabulary size       : {VOCAB_SIZE}")

print("\n" + "=" * 65)
print("  Preprocessing complete. Ready for training notebooks.")
print("=" * 65)